## 1. DCV_PROJECT

Mutual fund costs and returns. Our slice is the mid cap.

- **Category:** Mid Cap only. When checked the filters in the website AMFI writes this category two ways, Equity Scheme - Mid Cap Fund and Equity Schemes - Mid Cap Fund (with and without the s), and different fund houses are filed under different ones. We collect both.
- Mid cap means medium market capitalization 
- Market capitalization is the total market value of companys shares
- In India generally there are three categories of companies
- they are large cap ,mid cap and small cap 
- the large cap lies in the ranking of 1 to 100 ,the medium in the range of 101 to 250 and the small is from 251
- Our project is on midcap companies
- **Plans:** Direct and Regular, since the question is about the gap between them.
- Direct plan means we invest directly in the mutual fund company
- Regular Plan means we invest through a distributor,broker or agent.
- The ter is low for direct plan and generally high for regular plan.
- **Option:** Growth only. Dividend (IDCW) options drop in NAV when they pay out and that would look like a loss. We apply this in the clean step, not here.
- Options are those that determine how profits that are generated are managed
- Simply they are how the returns are handled
- In growth the returns are accumulated in the scheme
- In dividend scheme you receive some money and those are cutoff from the returns
- So it may seem sometimes that the nav is decreasing
- We will restrict our analysis to only growth options
- **Window:** We selected January 1, 2019, to October 31, 2026, as the collection period for NAV data. For TER, we use monthly data from January 2019 to September 2026,
- **Sources:** AMFI NAV history (semicolon separated text) and AMFI Total Expense Ratio (JSON API). Both were collected on the date in the file names.

**Project 11 from the sheet: Mutual fund costs and returns.**
- The question this project visuvalizes is Do equity funds with a higher expense ratio return less, and how large is the gap between direct and regular plans?
- NAV : Net Asset Value 
- It is the per share value of a mutual fund 
- TER - Total Expense Ratio
- It is the charges made by the company and may be the distributor to run and operate mutual fund

## 2. Data and methods

### 2.1 Data collection

### Sources

| Source | URL | Methods | Terms |
|---|---|---|---|
| AMFI NAV history | https://portal.amfiindia.com/DownloadNAVHistoryReport_Po.aspx | requests, semicolon separated text, no key | https://www.amfiindia.com/terms-of-use |
| AMFI Total Expense Ratio | https://www.amfiindia.com/api/populate-te-rdata-revised | requests, JSON API, one call per category, month and page | https://www.amfiindia.com/terms-of-use |

The below code creates three folders inside the data collection folder named raw,interim and clean in which the data directly scraped from the website is stored in the raw,
the intermediate files are stored in the interim and the cleaned files will be stored in the clean folder

In [1]:
import os
import re
import json
import time
import glob
import datetime
import collections
import requests
import numpy as np
import pandas as pd
from urllib import robotparser
from urllib.parse import urlparse, urljoin
from datetime import date
for directory in ["raw", "raw/nav", "raw/ter", "raw/robots", "interim", "clean"]:
    os.makedirs(directory, exist_ok=True)
TODAY = "2026-10-22"
print("Collected on:", TODAY)

MIDCAP_TER_CATS = [76, 17]
# We checked the ter filters and noticed that for equity schemes midcap we got 76 whereas for equity scheme - midcap we got 17 so we stored both and looped for both in the data 
# to find the total number of schemes
TER_URL = "https://www.amfiindia.com/api/populate-te-rdata-revised"
NAV_URL = "https://portal.amfiindia.com/DownloadNAVHistoryReport_Po.aspx"

Collected on: 2026-10-22


#### 2.2 Checking for robots.txt
<ul>
<li>We used both baseline urls from the given links and then used requests to get the response from those websites with robots.txt and then used robot parser to get the data of whether it is given permission legally 
<li>Robots.txt is just a permission instructions and it never stops us from scraping the website

In [3]:
host_urls = {"nav": "https://portal.amfiindia.com/DownloadNAVHistoryReport_Po.aspx?frmdt=01-Sep-2026&todt=03-Sep-2026","ter": "https://www.amfiindia.com/api/populate-te-rdata-revised?MF_ID=All&Month=08-2026&strCat=-1&strType=-1&page=1&pageSize=100"}
for name,host_url  in host_urls.items():
    p = urlparse(host_url)
    # using urllib to join both the urls
    robots_url = urljoin(f"{p.scheme}://{p.netloc}","/robots.txt")
    robots_path = f"raw/robots/amfi_{name}_robots_{TODAY}.txt"
    print(f"\n--- {name} ---")
    print("Source URL:", host_url)
    print("Robots URL:", robots_url)
    # So we check whether the requests succeeded actually or it returns an 400 code
    try:
        if os.path.exists(robots_path):
            # Checks if the file already existed so prevent multiple downloads
            print(f"[{name}] robots.txt already saved; skipping download.")
            print("Saved file:", robots_path)
        else:
            # If not exits
            response = requests.get(robots_url,timeout=30)
            print(f"[{name}] {robots_url} ,status code {response.status_code}")
            print("Content-Type:", response.headers.get("Content-Type"))
            print("Response:")
            print(response.text[:500])
            with open(robots_path,"w") as f:
                f.write(response.text)
            if response.status_code == 200:
                rp = robotparser.RobotFileParser()
                rp.parse(response.text.splitlines())
                print("Can fetch source URL:",rp.can_fetch("*",host_url))
            elif response.status_code in [401,403]:
                print("The server blocked the requests")
            elif response.status_code == 404:
                print("No robots.txt published")
            else:
                print("The scraping not succeeded")
    except requests.RequestException as error:
        print("Request failed:",repr(error))
    print("-"*60)


--- nav ---
Source URL: https://portal.amfiindia.com/DownloadNAVHistoryReport_Po.aspx?frmdt=01-Sep-2026&todt=03-Sep-2026
Robots URL: https://portal.amfiindia.com/robots.txt
[nav] robots.txt already saved; skipping download.
Saved file: raw/robots/amfi_nav_robots_2026-10-22.txt
------------------------------------------------------------

--- ter ---
Source URL: https://www.amfiindia.com/api/populate-te-rdata-revised?MF_ID=All&Month=08-2026&strCat=-1&strType=-1&page=1&pageSize=100
Robots URL: https://www.amfiindia.com/robots.txt
[ter] robots.txt already saved; skipping download.
Saved file: raw/robots/amfi_ter_robots_2026-10-22.txt
------------------------------------------------------------


Result

| Host | Used  | Status | What it says |
|---|---|---|---|
| www.amfiindia.com | TER API | 200 | Blocks /admin/, /login/ and /search/ for everyone. The API path we use is not on the list so can_fetch gave True.|
| portal.amfiindia.com | NAV download | 404 | No robots.txt on this host. We did not directly scraped and searched for the terms of service.|

The lines below come from the saved robots.txt files in raw/robots/

In [4]:
for name in ["ter", "nav"]:
    for path in sorted(glob.glob(f"raw/robots/amfi_{name}_robots_*.txt")):
        print(path)
        if name == "ter":
            for line in open(path, encoding="utf-8", errors="replace"):
                if line.lower().startswith(("user-agent", "disallow", "allow", "crawl-delay", "sitemap")):
                    print("   ", line.strip())
        else:
            print("    (a 404 page, this host has no robots.txt)")

raw/robots/amfi_ter_robots_2026-10-22.txt
    User-agent: *
    Disallow: /admin/
    Disallow: /login/
    Disallow: /search/
    User-agent: Screaming Frog SEO Spider
    Allow: /
    Sitemap: https://www.amfiindia.com/sitemap.xml
raw/robots/amfi_nav_robots_2026-10-22.txt
    (a 404 page, this host has no robots.txt)


### 2.3 Terms of use

Source: https://www.amfiindia.com/terms-of-use *(read on 2026-10-23)*

**Permitted uses:**
> *AMFI grants you a non-exclusive, personal, non-transferable, non-sublicensable, limited and revocable right to access, use and display this Site on any computers or other electronic display device of which you are a user, for your personal and non-commercial use only (the "Permitted Uses"). You agree to use the Site only for lawful purposes, for the Permitted Uses, and not for the Prohibited Uses. No other use of the Site is authorised unless you and we have agreed otherwise in writing.You may print copies of the Site's Content, provided that these copies are made only for the Permitted Uses and that you include any notices and any legal information contained in the Site Content, such as all copyright notices, trademark legends, or other proprietary rights notices as well as all legal caveats furnished in the footnotes or under "Important Legal Information" as shown on the screen or through a link. Limited linking to the Site is permitted if done in full compliance with all applicable laws, these Terms of Use.*

**Prohibited uses (storage):**
> *You are prohibited from any use of the Site that would give rise to liability or otherwise violate any applicable laws or regulations or the Terms of Use. You may not publicly perform, publicly display, transmit, publish, participate in the sale or transfer of, modify, or create derivative works based on anything available through the Site, in whole or in part. You shall not store electronically any significant portion of any part of the Site. No reverse engineering, linking, framing, or modification of any part of this Site is allowed without our written approval. Any modification or unauthorized use is a violation AMFI’s copyrights and other proprietary rights.*

The terms say nothing about automated requests, APIs or robots.txt.

- We only keep the mid cap slice and a limited window. The NAV download has no category filter, so the raw NAV files also hold the other categories of each fund house. They  cut down to mid cap in the interim notebook.
- Raw data is not republished. Only summaries and plots go in the report.
- Requests are spaced out, 1 second between calls.

#### 2.4 Sample data from the sheet links
We first downloaded the two sample links from the project sheet, just to see how each source is returning the data.We checked the format of the data and what it actualy contains.We store the raw data in raw/. We never edit raw files. Every change happens later in `interim/` and `clean/`.

- **NAV:** parameters frmdt and todt (DD-Mon-YYYY), semicolon separated text, no key.
- **TER:** parameters MF_ID, Month=MM-YYYY, strCat, strType, page, pageSize, JSON, no key.

In [5]:
print(host_urls["nav"])
os.makedirs("raw/check",exist_ok=True)

file_path = f"raw/check/amfi_nav_{TODAY}.txt"
if os.path.exists(file_path):
    print("File already saved")
    response_status = 200
else:
    response = requests.get(host_urls["nav"],timeout=30)
    response.raise_for_status()
    with open(file_path,"wb") as file:
        file.write(response.content)
    response_status = response.status_code
    print("NAV sample downloaded.")
print("NAV data extracted succesfully.")
print("HTTP status:",response_status)
print("File Saved:",file_path)
print("File Size:",os.path.getsize(file_path),"bytes")

https://portal.amfiindia.com/DownloadNAVHistoryReport_Po.aspx?frmdt=01-Sep-2026&todt=03-Sep-2026
File already saved
NAV data extracted succesfully.
HTTP status: 200
File Saved: raw/check/amfi_nav_2026-10-22.txt
File Size: 1624510 bytes


In [6]:
os.makedirs("raw/check",exist_ok = True)

file_path = f"raw/check/amfi_ter_{TODAY}.json"
if os.path.exists(file_path):
    print("File already exists")
    response_status = 200
    content_type = "application/json; charset=utf-8"
else:
    response = requests.get(host_urls["ter"],timeout = 30)
    response.raise_for_status()
    with open(file_path,"wb") as file:
        file.write(response.content)
    response_status = response.status_code
    content_type = response.headers.get("Content-Type")
    print("TER sample downloaded.")
print("TER data extracted.")
print("HTTP status:", response_status)
print("Content-Type:", content_type)
print("File saved:", file_path)
print("File size:", os.path.getsize(file_path), "bytes")
# THe below code checks whether the file is actually json or some other file
try:
    with open(file_path, encoding="utf-8") as f:
        ter_data = json.load(f)
    print("Valid JSON: Yes")
    print("Response type:",type(ter_data).__name__)
except ValueError:
    print("Valid JSON: No")
    with open(file_path, encoding="utf-8", errors="replace") as f:
        print(f.read(500))

File already exists
TER data extracted.
HTTP status: 200
Content-Type: application/json; charset=utf-8
File saved: raw/check/amfi_ter_2026-10-22.json
File size: 54128 bytes
Valid JSON: Yes
Response type: dict


#### 2.5 TER collection
- Lets see what the json actually contains, what are the keys in them ,how the meta data is stored and other properties of the data

In [7]:
with open(f"raw/check/amfi_ter_{TODAY}.json") as f:
    ter = json.load(f)
print(ter.keys())
print(json.dumps(ter, indent=2)[:2000])

dict_keys(['data', 'meta'])
{
  "data": [
    {
      "NSDLSchemeCode": "360O/O/H/BHF/23/07/0007",
      "Scheme_Name": "360 ONE Balanced Hybrid Fund",
      "SchemeType_Desc": "Open Ended",
      "SchemeCat_Desc": "Hybrid Schemes - Balanced Hybrid Fund",
      "TER_Year": "2026-2027",
      "TER_Date": "2026-08-01T00:00:00.000Z",
      "R_BER": "1.7800",
      "R_BrokerageCost": "0.0300",
      "R_TransactionCost": "0.0000",
      "R_StatutoryLevies": "0.3800",
      "R_TER": "2.1900",
      "D_BER": "0.4500",
      "D_BrokerageCost": "0.0300",
      "D_TransactionCost": "0.0000",
      "D_StatutoryLevies": "0.1400",
      "D_TER": "0.6200",
      "MF_ID": 62,
      "Month": "08-2026"
    },
    {
      "NSDLSchemeCode": "360O/O/H/BHF/23/07/0007",
      "Scheme_Name": "360 ONE Balanced Hybrid Fund",
      "SchemeType_Desc": "Open Ended",
      "SchemeCat_Desc": "Hybrid Schemes - Balanced Hybrid Fund",
      "TER_Year": "2026-2027",
      "TER_Date": "2026-08-02T00:00:00.000Z",
      "

In [8]:
# THis contains the meta data present in the json
print(ter["meta"])
print("records on this page:",len(ter["data"]))

{'page': 1, 'pageSize': 100, 'total': 63686, 'pageCount': 637}
records on this page: 100


In [9]:
# page is the current page it is in 
# page size is the total number of records that can be maximum extracted in the stage

### Picking the mid cap filter as the output gives all fund types

The unfiltered call gives 63,686 records for a single month. On the AMFI TER page (https://www.amfiindia.com/ter-of-mf-schemes) choosing Mid Cap sends strCat=76 and strType=1, so we send the same values. The server gives at most 100 records per page.

**Two mid cap categories we tols (equity scheme and equity schemes).** 
The strCat = 76 ressembles the equity schemes mid cap and the strCat =17 ressembles the equity scheme in the data

In [10]:
for cat in MIDCAP_TER_CATS:
    r = requests.get(TER_URL,
                     params={"MF_ID": "All", "Month": "08-2026", "strCat": cat,
                             "strType": 1, "page": 1, "pageSize": 100}, timeout=60)
    d = r.json()
    print("Category", cat)
    print(r.status_code, d["meta"], len(d["data"]))
    print("category:",{x["SchemeCat_Desc"] for x in d["data"]})
    print("schemes:", len({x["Scheme_Name"] for x in d["data"]}))
    print("dates:", len({x["TER_Date"] for x in d["data"]}))
    time.sleep(1)

Category 76
200 {'page': 1, 'pageSize': 100, 'total': 419, 'pageCount': 5} 100
category: {'Equity Schemes - Mid Cap Fund'}
schemes: 5
dates: 31
Category 17
200 {'page': 1, 'pageSize': 100, 'total': 600, 'pageCount': 6} 100
category: {'Equity Scheme - Mid Cap Fund'}
schemes: 4
dates: 31


To make sure no third mid cap id is hiding, the cell below runs `strCat` 1 to 200 again for Aug 2026 and keeps the ids whose category has mid cap in the name. The responses are saved in `raw/ter/discovery/`. It should give only 76 and 17. `Large & Mid Cap` also has mid cap in its name but it is a different SEBI category, so it is printed as left out.

**How we found the two mid cap category ids.** On the AMFI TER page (https://www.amfiindia.com/ter-of-mf-schemes) we checked the Category filter by hand. Choosing Mid Cap sends strCat=76, which is Equity Schemes - Mid Cap Fund (with an s). The same filter list also has Equity Scheme - Mid Cap Fund (without the s), which is strCat=17. Different fund houses are filed under different ones, so we collect both (MIDCAP_TER_CATS = [76, 17]). Large & Mid Cap is a separate SEBI category and is left out.

We did not scan every possible strCat number. We only used the ids that appear in the website filter.
*Manual step, as the rubric asks: page https://www.amfiindia.com/ter-of-mf-schemes, Category filter, Mid Cap entries (both spellings), viewed on 2026-10-24.*

### 3) Coverage of mid cap TER over time

The next cell counts the mid cap TER records in some random months, for both categories. A scheme has about one record per day, so records divided by the days in the month gives roughly the number of schemes reported.



In [11]:
for m in ["01-2019", "01-2022", "01-2024", "01-2025", "03-2026",
          "05-2026", "06-2026", "07-2026", "08-2026", "09-2026"]:
    combined = 0
    for cat in MIDCAP_TER_CATS:
        r = requests.get(TER_URL,
                         params={"MF_ID": "All", "Month": m, "strCat": cat, "strType": 1,
                                 "page": 1, "pageSize": 100}, timeout=60)
        d = r.json()
        names = {x["Scheme_Name"] for x in d["data"]}
        combined += d["meta"]["total"]
        print(m, "| cat", cat, "| total:", d["meta"]["total"], "| pages:", d["meta"]["pageCount"],
              "| names on page 1:", len(names))
        time.sleep(1)
    print(m, "| combined total:", combined)

01-2019 | cat 76 | total: 115 | pages: 2 | names on page 1: 4
01-2019 | cat 17 | total: 495 | pages: 5 | names on page 1: 4
01-2019 | combined total: 610
01-2022 | cat 76 | total: 139 | pages: 2 | names on page 1: 4
01-2022 | cat 17 | total: 522 | pages: 6 | names on page 1: 4
01-2022 | combined total: 661
01-2024 | cat 76 | total: 290 | pages: 3 | names on page 1: 4
01-2024 | cat 17 | total: 549 | pages: 6 | names on page 1: 4
01-2024 | combined total: 839
01-2025 | cat 76 | total: 294 | pages: 3 | names on page 1: 4
01-2025 | cat 17 | total: 563 | pages: 6 | names on page 1: 4
01-2025 | combined total: 857
03-2026 | cat 76 | total: 392 | pages: 4 | names on page 1: 4
03-2026 | cat 17 | total: 572 | pages: 6 | names on page 1: 4
03-2026 | combined total: 964
05-2026 | cat 76 | total: 391 | pages: 4 | names on page 1: 4
05-2026 | cat 17 | total: 594 | pages: 6 | names on page 1: 4
05-2026 | combined total: 985
06-2026 | cat 76 | total: 387 | pages: 4 | names on page 1: 4
06-2026 | cat 

THis is just the output stored in the table format
| Month | strCat=76 | strCat=17 | Combined |
|---|---|---|---|
| Jan 2019 | 115 | 495 | 610 |
| Jan 2022 | 139 | 522 | 661 |
| Jan 2024 | 290 | 549 | 839 |
| Jan 2025 | 294 | 563 | 857 |
| Mar 2026 | 392 | 572 | 964 |
| Aug 2026 | 419 | 600 | 1,019 |
| Sep 2026 | 411 | 581 | 992 |

Jan 2019 works out to about 20 schemes (610 / 31) and Aug 2026 to about 33 (1,019 / 31). So the number of mid cap schemes in the data grows over time. This is a coverage limitation as schemes increased .

All pages of every month are downloaded for both categories and saved in raw/ter/. The number of pages is read from meta.pageCount. File names look like amfi_ter_midcap_cat(id)_(month)_p(page)_(date).json. The helper ter_page() reuses a file that is already saved, so nothing is downloaded twice. The fund house ids (MF_ID) come from these records.

In [12]:
ter_log = []
for cat in MIDCAP_TER_CATS:
    for year in range(2019, 2027):
        # one folder per year, for example raw/ter/2019/
        year_folder = f"raw/ter/{year}"
        os.makedirs(year_folder, exist_ok=True)
        for month in range(1, 13):
            # TER data stops at September 2026
            if year == 2026 and month > 9:
                break
            month_text = f"{month:02d}-{year}"
            page = 1
            page_count = 1
            while page <= page_count:
                file_name = f"amfi_ter_midcap_cat{cat}_{month_text}_p{page}_{TODAY}.json"
                file_path = f"{year_folder}/{file_name}"
                saved_files = glob.glob(f"{year_folder}/amfi_ter_midcap_cat{cat}_{month_text}_p{page}_*.json")
                if saved_files:
                    file_path = saved_files[0]
                else:
                    response = requests.get(
                        TER_URL,
                        params={"MF_ID": "All", "Month": month_text, "strCat": cat,
                                "strType": 1, "page": page, "pageSize": 100},
                        timeout=60)
                    response.raise_for_status()
                    with open(file_path, "wb") as f:
                        f.write(response.content)
                    time.sleep(1)
                with open(file_path, encoding="utf-8") as f:
                    page_data = json.load(f)
                total = page_data["meta"]["total"]
                page_count = page_data["meta"]["pageCount"]
                ter_log.append((cat, month_text, page, total, len(page_data["data"])))
                page += 1
            print("Completed: cat", cat, month_text, "Records:", total, "Pages:", page_count)
ter_log_df = pd.DataFrame(ter_log, columns=["cat", "month", "page", "total", "records"])
ter_log_df.to_csv(f"raw/ter/amfi_ter_midcap_collection_log_{TODAY}.csv", index=False)
print("\nTER COLLECTION COMPLETED, page files in log:", len(ter_log_df))
print(ter_log_df.groupby(["cat", "month"])["total"].first().groupby("cat").sum())

Completed: cat 76 01-2019 Records: 115 Pages: 2
Completed: cat 76 02-2019 Records: 102 Pages: 2
Completed: cat 76 03-2019 Records: 101 Pages: 2
Completed: cat 76 04-2019 Records: 109 Pages: 2
Completed: cat 76 05-2019 Records: 115 Pages: 2
Completed: cat 76 06-2019 Records: 108 Pages: 2
Completed: cat 76 07-2019 Records: 118 Pages: 2
Completed: cat 76 08-2019 Records: 144 Pages: 2
Completed: cat 76 09-2019 Records: 137 Pages: 2
Completed: cat 76 10-2019 Records: 142 Pages: 2
Completed: cat 76 11-2019 Records: 136 Pages: 2
Completed: cat 76 12-2019 Records: 145 Pages: 2
Completed: cat 76 01-2020 Records: 144 Pages: 2
Completed: cat 76 02-2020 Records: 133 Pages: 2
Completed: cat 76 03-2020 Records: 143 Pages: 2
Completed: cat 76 04-2020 Records: 138 Pages: 2
Completed: cat 76 05-2020 Records: 140 Pages: 2
Completed: cat 76 06-2020 Records: 142 Pages: 2
Completed: cat 76 07-2020 Records: 145 Pages: 2
Completed: cat 76 08-2020 Records: 138 Pages: 2
Completed: cat 76 09-2020 Records: 141 P

While looking at the TER files we saw that `NSDLSchemeCode` is empty for most of the older records. The next cell calls the API again for one month to confirm it is the API and not our saving code. So we cannot use that column as a key. Schemes are identified by fund house, scheme name, category and date instead.

In [13]:
print(glob.glob("raw/ter/*12-2023_p3*"))
r = requests.get(TER_URL, params={"MF_ID": "All", "Month": "12-2023", "strCat": 76,
                                  "strType": 1, "page": 3, "pageSize": 100}, timeout=60)
print(r.status_code, r.text[:300])

[]
200 {"data":[{"NSDLSchemeCode":null,"Scheme_Name":"Mahindra Manulife Mid Cap Fund","SchemeType_Desc":"Open Ended","SchemeCat_Desc":"Equity Schemes - Mid Cap Fund","R_BaseTER":"1.9200","TER_Year":"2023-2024","TER_Date":"2023-12-07T00:00:00.000Z","R_6A_B":"0.0000","R_6A_C":"0.0500","R_GST":"0.0700","R_TER


Some fund houses have the same fund written two ways (`Mid Cap` and `Midcap`, for example JM, Kotak, ICICI Prudential, Invesco and Bandhan). The sheet says NAV and TER use different scheme keys, so we will match them by name later and need to handle these spellings in cleaning.

In [14]:
# Step 1: TER scheme names under each fund house (Aug 2026, both categories)
ter_names = {}
for cat in MIDCAP_TER_CATS:
    page, page_count = 1, 1
    while page <= page_count:
        r = requests.get(TER_URL, params={"MF_ID": "All", "Month": "08-2026", "strCat": cat,
                                          "strType": 1, "page": page, "pageSize": 100}, timeout=60)
        data = r.json()
        page_count = data["meta"]["pageCount"]
        for record in data["data"]:
            mf = int(record["MF_ID"])
            ter_names.setdefault(mf, set()).add(record["Scheme_Name"])
        page += 1
        time.sleep(1)
all_ids = sorted(ter_names)
# Step 2: NAV scheme names under a mid cap heading, same month (first week of Aug 2026)
nav_names = {}
fund_house_name = {}
for mf in all_ids:
    r = requests.get(NAV_URL, params={"mf": mf, "tp": 1, "frmdt": "01-Aug-2026", "todt": "07-Aug-2026"}, timeout=60)
    nav_names[mf] = set()
    fund_house_name[mf] = "name not found"
    heading = ""
    for line in r.text.splitlines()[1:]:
        line = line.strip()
        if line == "":
            continue
        parts = line.split(";")
        if parts[0].isdigit():
            text = heading.lower().replace("-", " ")
            if ("mid cap" in text or "midcap" in text) and "large" not in text:
                nav_names[mf].add(parts[1])
        elif line.startswith(("Open Ended", "Close Ended", "Interval")):
            heading = line
        else:
            fund_house_name[mf] = line
    time.sleep(1)

def tidy(name):
    return " ".join(name.lower().replace("-", " ").replace("mid cap", "midcap").split())

for mf in all_ids:
    nav_list = [tidy(n) for n in nav_names[mf]]
    ter_list = [tidy(n) for n in ter_names[mf]]
    not_in_nav = [n for n in ter_names[mf] if not any(tidy(n) in nav for nav in nav_list)]
    not_in_ter = [n for n in nav_names[mf] if not any(ter in tidy(n) for ter in ter_list)]
    print(mf, fund_house_name[mf], "| TER:", len(ter_list), "| not in NAV:", len(not_in_nav), "| not in TER:", len(not_in_ter))
    for n in not_in_nav:
        print("   only in TER:", n)
    for n in not_in_ter:
        print("   only in NAV:", n)

3 Aditya Birla Sun Life Mutual Fund | TER: 1 | not in NAV: 0 | not in TER: 0
4 Baroda BNP Paribas Mutual Fund | TER: 1 | not in NAV: 0 | not in TER: 0
6 DSP Mutual Fund | TER: 1 | not in NAV: 0 | not in TER: 0
9 HDFC Mutual Fund | TER: 1 | not in NAV: 0 | not in TER: 0
13 quant Mutual Fund | TER: 1 | not in NAV: 0 | not in TER: 0
16 JM Financial Mutual Fund | TER: 2 | not in NAV: 0 | not in TER: 0
17 Kotak Mahindra Mutual Fund | TER: 2 | not in NAV: 0 | not in TER: 0
18 LIC Mutual Fund | TER: 1 | not in NAV: 0 | not in TER: 0
20 ICICI Prudential Mutual Fund | TER: 2 | not in NAV: 0 | not in TER: 0
21 Nippon India Mutual Fund | TER: 1 | not in NAV: 0 | not in TER: 0
22 SBI Mutual Fund | TER: 1 | not in NAV: 0 | not in TER: 0
25 Tata Mutual Fund | TER: 1 | not in NAV: 0 | not in TER: 0
26 Taurus Mutual Fund | TER: 1 | not in NAV: 0 | not in TER: 0
27 Franklin Templeton Mutual Fund | TER: 1 | not in NAV: 0 | not in TER: 0
28 UTI Mutual Fund | TER: 1 | not in NAV: 0 | not in TER: 0
32 Cana

#### 4) NAV collection from 2019
- **URL:** https://portal.amfiindia.com/DownloadNAVHistoryReport_Po.aspx
- **Parameters:** mf (fund house id), tp=1 (open ended), frmdt and todt (DD-Mon-YYYY)
- AMFI allows only 90 days per download, so we go through the period in 90 day chunks, one fund house at a time.
- **There is no category filter for NAV.** Every file has all open ended schemes of that fund house, not only mid cap. We keep the raw files as they are and pick the mid cap rows out later.

**Cross-check on the 3 day sample.** AMFI spells the heading two ways here as well (Equity Schemes - Mid Cap Fund and Equity Scheme - Mid Cap Fund), so every filter uses the pattern mid[\s-]?cap (ignoring case) and drops headings with large. Large & Mid Cap is a separate category so we dont consider about that.

In [15]:
with_s = set()
without_s = set()
current_heading = ""
current_amc = ""

nav_file = sorted(glob.glob("raw/check/amfi_nav_*.txt"))[-1]
print("reading:", nav_file)

with open(nav_file, encoding="utf-8", errors="replace") as f:
    next(f)
    for line in f:
        line = line.strip()
        if line == "":
            continue
        first_field = line.split(";")[0]
        if first_field.isdigit():
            text = current_heading.lower()
            is_mid_cap = ("mid cap" in text.replace("-", " ") or "midcap" in text) and "large" not in text
            if is_mid_cap:
                if "equity schemes" in text:
                    with_s.add(current_amc)
                elif "equity scheme" in text:
                    without_s.add(current_amc)
        elif line.startswith(("Open Ended", "Close Ended", "Interval")):
            current_heading = line
        else:
            current_amc = line

all_mid_cap_amcs = with_s | without_s
in_both = with_s & without_s

print("Fund houses under 'Equity Schemes - Mid Cap Fund' (with s):", len(with_s))
print("Fund houses under 'Equity Scheme - Mid Cap Fund' (no s):", len(without_s))
print("\nFund houses ONLY under the no-s heading:")
for name in sorted(without_s - with_s):
    print("  ", name)
print("\nFund houses under both spellings:", sorted(in_both))
print("\nTOTAL fund houses with a mid cap fund:", len(all_mid_cap_amcs))

reading: raw/check/amfi_nav_2026-10-22.txt


Fund houses under 'Equity Schemes - Mid Cap Fund' (with s): 14
Fund houses under 'Equity Scheme - Mid Cap Fund' (no s): 20

Fund houses ONLY under the no-s heading:
   Aditya Birla Sun Life Mutual Fund
   Axis Mutual Fund
   Baroda BNP Paribas Mutual Fund
   DSP Mutual Fund
   Edelweiss Mutual Fund
   Franklin Templeton Mutual Fund
   HDFC Mutual Fund
   ITI Mutual Fund
   LIC Mutual Fund
   Motilal Oswal Mutual Fund
   Nippon India Mutual Fund
   PGIM India Mutual Fund
   SBI Mutual Fund
   Sundaram Mutual Fund
   Tata Mutual Fund
   Taurus Mutual Fund
   Trust Mutual Fund
   UTI Mutual Fund
   Union Mutual Fund
   quant Mutual Fund

Fund houses under both spellings: []

TOTAL fund houses with a mid cap fund: 34


**Chunks and download loop.** 01-Jan-2019 to 31-Oct-2026 is split into 90 day chunks (32 chunks). The fund houses are the MF_ID's found in the TER files. The end date is fixed so the chunks (and file names) are the same on every run. Each response is saved unchanged as amfi_nav_mf(id)_(from)_(to)_(date).txt. A file that is already saved is skipped, so if the loop stops halfway we can run it again. There is a 1 second pause between calls.

In [16]:
# Step 1: fund house ids from the TER files
mf_ids = set()
for path in glob.glob("raw/ter/*/amfi_ter_midcap_cat*_p*_*.json"):
    with open(path, encoding="utf-8") as f:
        for record in json.load(f)["data"]:
            mf_ids.add(int(record["MF_ID"]))
mf_ids = sorted(mf_ids)
print(len(mf_ids), "fund houses:", mf_ids)

# Step 2: list every day, then cut the list into pieces of 90 days
all_days = pd.date_range("2019-01-01", "2026-9-30")
chunks = []
for i in range(0, len(all_days), 90):
    piece = all_days[i:i + 90]
    chunks.append((piece[0], piece[-1]))
print(len(chunks), "chunks,", len(chunks) * len(mf_ids), "calls")

# Step 3: download every chunk for every fund house
nav_log = []
new_downloads = 0
for mf in mf_ids:
    folder = f"raw/nav/mf{mf}"
    os.makedirs(folder, exist_ok=True)
    for first_day, last_day in chunks:
        from_text = first_day.strftime("%d-%b-%Y")
        to_text = last_day.strftime("%d-%b-%Y")
        name = f"amfi_nav_mf{mf}_{from_text}_{to_text}"
        saved = glob.glob(f"{folder}/{name}_*.txt")
        if saved:
            path = saved[0]
        else:
            path = f"{folder}/{name}_{TODAY}.txt"
            response = requests.get(NAV_URL, params={"mf": mf, "tp": 1,
                                                     "frmdt": from_text, "todt": to_text}, timeout=60)
            response.raise_for_status()
            with open(path, "wb") as f:
                f.write(response.content)
            new_downloads += 1
            time.sleep(1)
        nav_log.append((mf, from_text, to_text, os.path.getsize(path)))
    print("done fund house", mf)
nav_log_df = pd.DataFrame(nav_log, columns=["mf_id", "from", "to", "bytes"])
nav_log_df.to_csv(f"raw/nav/amfi_nav_collection_log_{TODAY}.csv", index=False)
print("Collection date:", TODAY)
print("Files downloaded in this run:", new_downloads)
print("Fund houses:", nav_log_df["mf_id"].nunique())
print("Files:", len(nav_log_df))
print("Total bytes:", nav_log_df["bytes"].sum())

34 fund houses: [3, 4, 6, 9, 13, 16, 17, 18, 20, 21, 22, 25, 26, 27, 28, 32, 33, 37, 42, 45, 46, 47, 48, 53, 55, 58, 61, 69, 70, 71, 72, 74, 76, 83]
32 chunks, 1088 calls
done fund house 3
done fund house 4
done fund house 6
done fund house 9
done fund house 13
done fund house 16
done fund house 17
done fund house 18
done fund house 20
done fund house 21
done fund house 22
done fund house 25
done fund house 26
done fund house 27
done fund house 28
done fund house 32
done fund house 33
done fund house 37
done fund house 42
done fund house 45
done fund house 46
done fund house 47
done fund house 48
done fund house 53
done fund house 55
done fund house 58
done fund house 61
done fund house 69
done fund house 70
done fund house 71
done fund house 72
done fund house 74
done fund house 76
done fund house 83
Collection date: 2026-10-22
Files downloaded in this run: 0
Fund houses: 34
Files: 1088
Total bytes: 1593091475


#### 4.1 Checks on the raw files
**TER.** Did every category and month return all its records, are there duplicate scheme-days, and do the TER fund houses match the ones we downloaded NAV for? The key for a duplicate is fund house + scheme name + category + date, because `NSDLSchemeCode` is empty for most older records and cannot be used as a key.

**NAV.** For every saved file we count the data rows, the mid cap rows and look at the header line. Files with no rows belong to fund houses that had nothing in that 90 day chunk, for example a fund that started late.

In [17]:
rows = []
for path in sorted(glob.glob("raw/ter/*/amfi_ter_midcap_*_p*_*.json")):
    with open(path, encoding="utf-8") as f:
        rows += json.load(f)["data"]
ter_all = pd.DataFrame(rows)
print("records read:", len(ter_all))
check = ter_log_df.groupby(["cat", "month"]).agg(total=("total", "first"), got=("records", "sum"))
print("category-months where records read differ from the total:", (check["total"] != check["got"]).sum())
key = ["MF_ID", "Scheme_Name", "SchemeCat_Desc", "TER_Date"]
print("repeated scheme-days (only counted, not removed):", ter_all.duplicated(key).sum())
print("records with empty NSDLSchemeCode:", ter_all["NSDLSchemeCode"].isna().sum(), "of", len(ter_all))
ter_ids = set(ter_all["MF_ID"].astype(int))
print("fund houses in TER but not in the NAV download:", sorted(ter_ids - set(mf_ids)))
print(ter_all["SchemeCat_Desc"].value_counts())

records read: 70019
category-months where records read differ from the total: 0
repeated scheme-days (only counted, not removed): 0
records with empty NSDLSchemeCode: 48482 of 70019
fund houses in TER but not in the NAV download: []
SchemeCat_Desc
Equity Scheme - Mid Cap Fund     48946
Equity Schemes - Mid Cap Fund    21073
Name: count, dtype: int64


In [18]:
report = []
for path in sorted(glob.glob("raw/nav/mf*/amfi_nav_mf*.txt")):
    mf = int(path.split("/")[2].replace("mf", ""))
    heading = ""
    all_rows = 0
    mid_cap_rows = 0
    with open(path, encoding="utf-8", errors="replace") as f:
        header = f.readline().strip()
        for line in f:
            line = line.strip()
            if line == "":
                continue
            if line.split(";")[0].isdigit():
                all_rows += 1
                text = heading.lower().replace("-", " ")
                if ("mid cap" in text or "midcap" in text) and "large" not in text:
                    mid_cap_rows += 1
            elif line.startswith(("Open Ended", "Close Ended", "Interval")):
                heading = line
    report.append((mf, os.path.basename(path), header[:70], all_rows, mid_cap_rows))
rep = pd.DataFrame(report, columns=["mf", "file", "header", "rows", "mid_rows"])
print("fund houses:", rep["mf"].nunique(), "| files:", len(rep))
print(rep["header"].value_counts())
print("empty files:", (rep["rows"] == 0).sum())

fund houses: 34 | files: 1088
header
Scheme Code;NAV Name;Plan;Option;ISIN Div Payout/ISIN Growth;ISIN Div     1021
                                                                            67
Name: count, dtype: int64
empty files: 67


#### 5) RAW FILES
For each kind of raw file we see its shape, dtypes, a few rows and what each column means. 
NAV file. -  One chunk file of one fund house. The raw file is plain text, so every column is read as text. We only take the lines with 8 fields (the heading and fund house lines are left out for this table).

In [20]:
mf_demo = 3 if 3 in mf_ids else mf_ids[0]
nav_path = sorted(glob.glob(f"raw/nav/mf{mf_demo}/*.txt"))[-1]
rows = []
with open(nav_path, encoding="utf-8", errors="replace") as f:
    header = f.readline().strip().split(";")
    for line in f:
        fields = line.strip().split(";")
        if len(fields) == len(header) and fields[0].isdigit():
            rows.append(fields)
nav_one = pd.DataFrame(rows, columns=header)
print(nav_path)
print("shape:", nav_one.shape)
print(nav_one.dtypes)
nav_one.head()

raw/nav/mf3/amfi_nav_mf3_30-Nov-2024_27-Feb-2025_2026-10-22.txt
shape: (26679, 8)
Scheme Code                    str
NAV Name                       str
Plan                           str
Option                         str
ISIN Div Payout/ISIN Growth    str
ISIN Div Reinvestment          str
Net Asset Value                str
Date                           str
dtype: object


,Scheme Code,NAV Name,Plan,Option,ISIN Div Payout/ISIN Growth,ISIN Div Reinvestment,Net Asset Value,Date
0,148921,Aditya Birla Sun Life Multi-Cap Fund-Direct Gr...,Direct Plan,GROWTH,INF209KB1Y49,,20.67,02-Dec-2024
1,148921,Aditya Birla Sun Life Multi-Cap Fund-Direct Gr...,Direct Plan,GROWTH,INF209KB1Y49,,20.85,03-Dec-2024
2,148921,Aditya Birla Sun Life Multi-Cap Fund-Direct Gr...,Direct Plan,GROWTH,INF209KB1Y49,,20.94,04-Dec-2024
3,148921,Aditya Birla Sun Life Multi-Cap Fund-Direct Gr...,Direct Plan,GROWTH,INF209KB1Y49,,20.98,05-Dec-2024
4,148921,Aditya Birla Sun Life Multi-Cap Fund-Direct Gr...,Direct Plan,GROWTH,INF209KB1Y49,,21.02,06-Dec-2024


| Column | Meaning |
|---|---|
| Scheme Code | One Record Id |
| NAV Name | scheme name as AMFI writes it, with plan and option in the name |
| Plan | Direct Plan or Regular Plan( Intermediate agent or direct investment in company) |
| Option | Growth, IDCW (dividend) payout or reinvestment(the profits are manually added or the returns are distributed) |
| ISIN Div Payout/ISIN Growth |It acts like passport number for each fund making sure each fund is tracked |
| ISIN Div Reinvestment | This is the specific tracking code ISIN assigned to your units because you are enrolled in the automatic dividend reinvestment plan|
| Net Asset Value | price of one unit or share  |
| Date | date of the NAV,DD-Mon-YYYY|

The category (for example `Equity Scheme - Mid Cap Fund`) and the fund house are not columns. They are lines between the rows. We keep that information in the interim notebook.

**TER file.** One page of category 17 for Aug 2026 (new column layout) and one page of Jan 2019 (old layout). One record is one scheme on one day.

In [24]:
ter_new = pd.DataFrame(json.load(open(sorted(glob.glob("raw/ter/2026/amfi_ter_midcap_cat17_08-2026_p1_*.json"))[0], encoding="utf-8"))["data"])
ter_old = pd.DataFrame(json.load(open(sorted(glob.glob("raw/ter/2019/amfi_ter_midcap_cat17_01-2019_p1_*.json"))[0], encoding="utf-8"))["data"])
for label, d in [("Aug 2026", ter_new), ("Jan 2019", ter_old)]:
    print(label, "shape:", d.shape)
    print(d.dtypes)
    print()
ter_new.head(3)

Aug 2026 shape: (100, 18)
NSDLSchemeCode         str
Scheme_Name            str
SchemeType_Desc        str
SchemeCat_Desc         str
TER_Year               str
TER_Date               str
R_BER                  str
R_BrokerageCost        str
R_TransactionCost      str
R_StatutoryLevies      str
R_TER                  str
D_BER                  str
D_BrokerageCost        str
D_TransactionCost      str
D_StatutoryLevies      str
D_TER                  str
MF_ID                int64
Month                  str
dtype: object

Jan 2019 shape: (100, 18)
NSDLSchemeCode     object
Scheme_Name           str
SchemeType_Desc       str
SchemeCat_Desc        str
R_BaseTER             str
TER_Year              str
TER_Date              str
R_6A_B                str
R_6A_C                str
R_GST                 str
R_TER                 str
D_BaseTER             str
D_6A_B                str
D_6A_C                str
D_GST                 str
D_TER                 str
MF_ID               int64
Month

,NSDLSchemeCode,Scheme_Name,SchemeType_Desc,SchemeCat_Desc,TER_Year,TER_Date,R_BER,R_BrokerageCost,R_TransactionCost,R_StatutoryLevies,R_TER,D_BER,D_BrokerageCost,D_TransactionCost,D_StatutoryLevies,D_TER,MF_ID,Month
0,ABSL/O/E/MIF/02/09/0017,Aditya Birla Sun Life MIDCAP Fund,Open Ended,Equity Scheme - Mid Cap Fund,2026-2027,2026-08-01T00:00:00.000Z,1.5500,0.0700,0.0000,0.4200,2.0400,0.8700,0.0700,0.0000,0.3100,1.2500,3,08-2026
1,ABSL/O/E/MIF/02/09/0017,Aditya Birla Sun Life MIDCAP Fund,Open Ended,Equity Scheme - Mid Cap Fund,2026-2027,2026-08-02T00:00:00.000Z,1.5500,0.0700,0.0000,0.4200,2.0400,0.8700,0.0700,0.0000,0.3100,1.2500,3,08-2026
2,ABSL/O/E/MIF/02/09/0017,Aditya Birla Sun Life MIDCAP Fund,Open Ended,Equity Scheme - Mid Cap Fund,2026-2027,2026-08-03T00:00:00.000Z,1.5500,0.0700,0.0000,0.4200,2.0400,0.8600,0.0700,0.0000,0.3100,1.2400,3,08-2026


| Column | Meaning |
|---|---|
| NSDLSchemeCode | Unique identifier assigned to a mutual fund scheme by NSDL |
| Scheme_Name | Name of the mutual fund scheme |
| SchemeType_Desc | Open Ended(A mutual fund scheme that allows investors to buy and redeem units at any time at any prevailing NAV) |
| SchemeCat_Desc | Description of type of mutual fund scheme(here equity scheme- midcap)(equity scheme- a mutual fund scheme that primarly invest in shares to generate long term capital)(midcap - a mutual fund category that primarly invests in medium sized companies) |
| TER_Year, TER_Date, Month | financial year for which TER is recorded , Date on TER Data was recorded , Month on which TER month was recorded |
| MF_ID | fund house id, same number as `mf` in the NAV request |
| R_TER, D_TER | total expense ratio of the Regular and the Direct plan, in % of average assets per year |
| R_BER, D_BER | base expense ratio |
| R_BrokerageCost, D_BrokerageCost | brokerage part(brokerage expenses incurred by a regular /direct mutual fund plan)|
| R_TransactionCost, D_TransactionCost | transaction cost |
| R_StatutoryLevies, D_StatutoryLevies | statutory levies part |
- Old part

| Column | Meaning |
|---|---|
| R_BaseTER, D_BaseTER | base expense ratio |
| R_6A_B, R_6A_C, D_6A_B, D_6A_C | extra charges allowed under SEBI regulation 52(6A) (old layout) |
| R_GST, D_GST | GST part (old layout) |

All numbers arrive as text such as `"2.1900"`. The old and the new layout have different parts but both have `R_TER` and `D_TER`.